# Intro til unntak

Vi vi ta en første kikk på unntak i uke 40

Vi vil snakke mer om unntak i uke 42, sammen med lesing fra fil.

## Unntak (kjøretidsfeil) - introduksjon

Tre typer feil:

1. syntaksfeil - ugyldig skrevet kode

2. logiske feil, men koden gjør ikke det den "skal",
    - ingen feilmelding fra Python, vet jo ikke hva du vil

3. kjøretidsfeil - kode som utøfrer ugyldig operasjon
    - dele på null
    - indeksere utenfor listen
    - konvertere ugyldig data
    - bruke variabelnavn eller funksjon som ikke er definert ennå
    - og mange flere

Hva kan vi gjøre med disse feilene?

1. Syntaksfeil - fiks koden!

2. Logiske feil - debug, analyser koden, finn ut hva den faktisk gjør
    - endre slik at det blir riktig 

3. Her skal vi fokusere på punkt 3, kjøretidsfeil, og se hvilke muligheter vi har

### Filosofisk spørsmål:

Er det noe vits å håndtere feil, hvis en ikke vet hva en skal gjøre med feilen?

## To feilhåndteringsfilosofier

### 1. Å sjekke PÅ FORHÅND slik at vi ikke vil få en feil-situasjon. F.eks.

- sjekke at inndata er gyldig heltall før vi utfører int(inndata)
- sjekke at vi ikke deler på null
- sjekke at indeks er gyldig før vi utfører liste[indeks]

### 2. Utfør koden (try), men sette opp et sikkerhetsnett for å "fange opp" feilsituasjonen.

Python lar oss sette opp et slikt nett, hvor vi kan skrive kode som håndterer
feilen, eller som vi også sier, `unntaket`.

Vi ser på hvordan denne mekanismen ser ut i et eksempel:

In [ ]:
try:
    print(1/1)
    print(1/0)
    print(2/1)

except ZeroDivisionError:
    print("Udefinert: Du kan ikke dele på null!")

print("Programmet fortsetter her.")

### Observasjoner:

Koden i `try`-blokken utføres, helt til det oppstår en feil (ZeroDivisionError)
- Programflyten hopper da ut av try-blokken og utfører første except-blokk som sier den håndterer den type feil.


### Eksempel

La oss lage en funksjon som ber om et flyttall fra brukeren,
og returnere det som flyttall (ikke tekst).

In [ ]:
def be_om_flyttall(prompt = "Skriv inn et flyttall: "):
    while True:
        try:
            tall = float(input(prompt))
            return tall
        except ValueError:
            print("Ugyldig input. Vennligst skriv inn et gyldig flyttall.")

# Her skal vi bruke funksjonen be_om_flyttall for å be brukeren om to tall,
# og deretter dele dem på hverandre. Vi håndterer også unntaket for deling på null.
try:
    teller = be_om_flyttall("Skriv inn teller: ")
    nevner = be_om_flyttall("Skriv inn nevner: ")
    resultat = teller / nevner
    print(f"Resultatet av {teller} / {nevner} er {resultat}.")
except ZeroDivisionError:
    print("Udefineret: Du kan ikke dele på null!")

## `raise` : Å utløse unntak ("throw exception")

float()-funksjonen prøver å konvertere det den får til et flyttall. Hvis den får input som
den ikke er programmert for å håndtere, f.eks. `float("tre")` så bruker den kommandoen `raise`

Vi kan selv bruke `raise` i egen kode, for å varsle om en feil-situasjon, f.eks. slik:

In [ ]:
def ikke_ferdig():
    raise NotImplementedError("Denne funksjonen er ikke ferdig implementert ennå.")

ikke_ferdig()

## Unntakstyper og dens hiarki

De forskjellige unntakstypene danner et hiarki. Så f.eks. en ZeroDivisionError er også en ArithmeticError, som også er av typen Exception. Vi skal bruke de under `Exception`.  

BaseException
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
└── Exception
    ├── ArithmeticError
    │   └── ZeroDivisionError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── TypeError
    ├── ValueError
    ├── NameError
    │   └── UnboundLocalError
    ├── AttributeError
    ├── FileNotFoundError
    └── OSError

## Hvilke type unntak skal jeg "fange"?

En kan bruke en mer generell unntakstype for å håndtere (fange) flere typer feil i samme `except`-blokk.

**NB: Det anbefales å bruke så spesifikk type som mulig, slik at vi kan håndtere feilen på riktig måte, og at 
det er klart hva en except-blokk er ment å håndtere**

Sammenlign de følgende: 

In [ ]:
# Her er et eksempel på hvordan man kan håndtere flere typer unntak i en enkelt try-blokk.
# Den første feilen som oppstår blir håndtert av `except`-blokken for den feiltypen,
# og kun denne ene. 
try:
    x, y = input("x,y = ").split(",")
    x = float(x)
    y = float(y)
    print(x / y)
except TypeError:
    print("TypeError: Du må skrive inn to tall separert med komma.")
except ValueError:
    print("ValueError: Du må skrive inn to gyldige tall.")
except ZeroDivisionError:
    print("ZeroDivisionError: Du kan ikke dele på null.")
    


In [ ]:
# Her er et eksempel på vi bruker en mer generell unntakstype.
# Alt som går galt i try-blokken vil bli fanget opp av denne `except`-blokken.
try:
    x, y = input("x,y = ").split(",")
    x = float(x)
    y = float(y)
    print(x / y)
except Exception as e:  # Vi kan fange opp alle typer unntak. Vi kan også få tilgang til feilmeldingen.
    print(f"En feil oppstod: {e}.")

## Håndter unntak der hvor en vet hva en vil gjøre med det

- Mange funksjoner kan "utløse" unntak, slik som float().
- Hva en gjør med feilen, overlates til brukeren av funksjonen. 
- Dette er et godt design-prinsipp: Funksjonen gjør "én ting":
    - lag flyttall
    - hva som skal skje hvis det ikke går, er ikke float() sin oppgave/problem.

MEN: Kanskje vi IKKE vil håndtere feilen et annet sted enn der vi kaller funksjonen som gir feil?

For å forstå hva som skjer med feil, skal vi først se på programflyten når vi kaller funksjoner

## Om kall-stakken (call stack)

Når vi kaller en funksjon, så

- tilordnes argumentene til input-parametrene
- funksjonskoden utføres
- ved `return` eller etter siste kodeline i funksjonen, så returnerer programflyten og fortsetter der funksjonen ble kalt.
- retur-verdi erstatter funksjonskall-uttrykket (float("3.15") blir erstattet av verdien 3.15)

Hvis en gjør et funksjonskall inne fra funksjonen, så skjer akkurat det samme:

- overfør argument til parametre
- utfør funskjonskoden 
- returner til der hvor kallet ble gjort

Disse lagene eller nivåene kalles for "call stack", og vi kan se dem når vi kjører koden i debug-modus.

Du kan også se call-stacken i feil-meldingen du får nå du får en feil

## Programflyten ved et unntak

Når det utløses et unntak, så vil vil programflyten flyttes til den nærmeste except-blokken i kall-stakken, som
spesifiserer den gitt feil-typen eller en mer generell feil-type. 

Sett break-punkt i print()-setningen under, og

In [ ]:
def f1(n):
    if n >= 10:
        raise ValueError("n må være i området 0-9.")
    return ["null","en", "to", "tre", "fire", "fem", "seks", "sju", "åtte", "ni"][n]

def f2(n):
    return "-".join([f1(i) for i in range(n)])

def f3(n):
    try:
        return f2(n)
    except Exception as e:
        print(f"En feil oppstod i f2: {e}.")
        return None

print(f3(5))  # Dette vil fungere fint og returnere "null-en-to-tre-fire"
print(f3(15))  # Dette vil kaste en ValueError i f, som blir fanget opp i h, og returnere None.

## Hvorfor ikke bare bruke if-setninger i stedet?
I mange tilfeller kan if-setninger brukes i stedet for unntaksbehandling.
- if-setninger: sjekke på forhånd så vi er sikre på at en operasjon går bra å utføre, deretter utføre operasjonen
- unntaksbehandling: prøve å utføre operasjonen (uten å ha sjekket), gjøre en alternativ handling hvis operasjonen feilet

MEN: ofte blir unntaksbehandling enklere, som illustrert med eksempler under.

Første eksempel: Her har vi verken if-setning eller unntak
- koden vil feile (ValueError) hvis bruker skriver noe som ikke er et tall

In [ ]:
tall = int(input("Skriv et heltall: "))
print("Tallet var", tall)

Andre eksempel: Fikser problemet med if-setning
- unngår ValueError
- men vil si "ikke et lovlig heltall" for negativt tall - var dette meningen?

In [ ]:
streng = input("Skriv et heltall: ")
if streng.isdigit():
    tall = int(streng)
    print("Tallet var", tall)
else:
    print("Du skrev ikke et lovlig heltall")

Tredje eksempel: Fortsatt if-setning
- lager koden litt mer avansert for å håndtere noen spesielle case
    - at tallet kan begynne med minustegn
    - at det kan være blanke tegn foran og bak tallet

In [ ]:
streng = input("Skriv et heltall: ")
s = streng.strip()
if (s[0].isdigit() or s[0] == '-') and s[1:].isdigit():
    tall = int(s)
    print("Tallet var", tall)
else:
    print("Du skrev ikke et lovlig heltall")

Koden over viser hva som av og til skjer når vi skal bruke if-setninger for å sjekke at det er trygt å utføre en viss operasjon (som her int(s)): if-setningen blir litt komplisert pga mange spesialcase. Her blir unntakshåndtering enklere kode:
- prøver å konvertere til int() og printe
- hvis det ikke gikk bra, gjør vi i stedet en alternativ print

In [ ]:
try:
    tall = int(input("Skriv et heltall: "))
    print("Tallet var", tall)
except:
    print("Du skrev ikke et lovlig heltall")

Fordelen med unntakshåndtering kan bli enda større hvis det er mange ulike ting som kan gå galt. Koden nedenfor har fire tilfeller av inntastede tall som skal konverteres med int(), og som hver for seg kan gå galt.

Det første eksemplet har verken if eller unntak, vil feile ved gal input.

In [ ]:
# UTEN FEILHÅNDTERING
input("V0 - uten feilhåndtering")
print('Alle data skal gis som heltall.')
alder = int(input('Oppgi alder: '))
vekt = int(input('Oppgi vekt i kg: '))
h = int(input('Oppgi høyde i cm: '))
epler = int(input('Hvor mange epler har du? '))

Neste eksempel prøver å samle opp alt som kan gå galt i én if-setning.
- fordel: unngår at koden blir veldig lang
- ulempe: gir ikke feilmeldingen til bruker på riktig sted. 
    - Selv om brukeren taster feil på alder, 
    - vil programmet fortsette å spørre om de andre opplysningene, 
    - og så gi feilmelding når alt er tastet inn

In [ ]:
# FEILHÅNDTERING MED if-else
input("V1 - if oppsamlet")
print('Alle data skal gis som heltall.')
alder = input('Oppgi alder: ')
vekt = input('Oppgi vekt i kg: ')
h = input('Oppgi høyde i cm: ')
epler = input('Hvor mange epler har du? ')
if (alder.isdigit() and vekt.isdigit()
    and h.isdigit() and epler.isdigit()):
    alder = int(alder)
    vekt = int(vekt)
    h = int(h)
    epler = int(epler)
else:
    print('Feil format på inndata')

Neste versjon klarer å gi feilmelding på rett tidspunkt
- ved å ha en if-setning for hver eneste input
- men dette gir lang og omstendelig kode

In [ ]:
# FEILHÅNDTERING MED if-else
input("V2 - if en og en")
print('Alle data skal gis som heltall.')
alder = input('Oppgi alder: ')
if alder.isdigit():
    alder = int(alder)
else:
    print('Feil format på inndata')
vekt = input('Oppgi vekt i kg: ')
if vekt.isdigit():
    vekt = int(vekt)
else:
    print('Feil format på inndata')
h = input('Oppgi høyde i cm: ')
if h.isdigit():
    h = int(h)
else:
    print('Feil format på inndata')
epler = input('Hvor mange epler har du? ')
if epler.isdigit():
    epler = int(epler)
else:
    print('Feil format på inndata')

Med unntakshåndtering (try-except) oppnår vi
- kortere og enklere kode
- feilmelding kommer på rett tidspunkt

In [ ]:
# FEILHÅNDTERING MED try-except
# Kortere og enklere kode, og gir beskjed om feil på rett tidspunkt
input("try except")
print('Alle data skal gis som heltall.')
try:
    alder = int(input('Oppgi alder: '))
    vekt = int(input('Oppgi vekt i kg: '))
    h = int(input('Oppgi høyde i cm: '))
    epler = int(input('Hvor mange epler har du? '))
except:  # Vi kan også droppe å skrive Exception, 
    print('Feil format på inndata')